In [1]:
import os
import re
import numpy as np
from scipy import sparse
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.decomposition import LatentDirichletAllocation

In [2]:
# read in marker labels
marker_labels = pd.read_csv("classical_ts__s__gmm_clustered__visible_signal.csv", index_col=0)

display(marker_labels)

,labels,probabilities
10001_at,7,0.857061
10003_f_at,3,0.999703
10005_at,3,0.997951
10010_at,5,0.986816
10011_at,7,0.701117
...,...,...
9992_at,0,0.993806
9994_at,3,0.998811
9996_at,2,0.966032
9998_at,2,0.993418


In [3]:
# read in metadata
metadata = pd.read_csv(os.path.join('..', 'data', 'marker_metadata.txt'), delimiter='\t', skiprows=17, index_col=0)
metaidx = metadata.index.tolist()
getidx = [idx for idx in metaidx if "AFFX" != idx[:4]]
metadata = metadata.loc[getidx].copy()
display(metadata)

,ORF,SPOT_ID,Species Scientific Name,Annotation Date,Sequence Type,Sequence Source,Target Description,Representative Public ID,Gene Title,Gene Symbol,ENTREZ_GENE_ID,RefSeq Transcript ID,SGD accession number,Gene Ontology Biological Process,Gene Ontology Cellular Component,Gene Ontology Molecular Function
ID,,,,,,,,,,,,,,,,
10000_at,YLR331C,NaN,Saccharomyces cerevisiae,"Oct 6, 2014",Exemplar sequence,Saccharomyces Genome Database,YLR331C questionable ORF,YLR331C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10001_at,YLR332W,NaN,Saccharomyces cerevisiae,"Oct 6, 2014",Exemplar sequence,Saccharomyces Genome Database,YLR332W Protein required for mating,YLR332W,O-glycosylated plasma membrane protein; acts a...,MID2,851042,NM_001182221,S000004324,0000767 // cell morphogenesis involved in conj...,0005886 // plasma membrane // inferred from el...,0004888 // transmembrane signaling receptor ac...
10002_i_at,YLR333C,NaN,Saccharomyces cerevisiae,"Oct 6, 2014",Exemplar sequence,Saccharomyces Genome Database,YLR333C Ribosomal protein S25B (S31B) (rp45) (...,YLR333C,ribosomal 40S subunit protein S25B,RPS25B,851045,NM_001182222,NaN,0002181 // cytoplasmic translation // inferred...,0005737 // cytoplasm // inferred from electron...,0003735 // structural constituent of ribosome ...
10003_f_at,YLR333C,NaN,Saccharomyces cerevisiae,"Oct 6, 2014",Exemplar sequence,Saccharomyces Genome Database,YLR333C Ribosomal protein S25B (S31B) (rp45) (...,YLR333C,Protein component of the small (40S) ribosomal...,RPS25B,851045,NM_001182222,S000004325,0002181 // cytoplasmic translation // inferred...,0005737 // cytoplasm // inferred from electron...,0003735 // structural constituent of ribosome ...
10004_at,YLR334C,NaN,Saccharomyces cerevisiae,"Oct 6, 2014",Exemplar sequence,Saccharomyces Genome Database,YLR334C questionable ORF,YLR334C,NaN,NaN,NaN,NaN,NaN,NaN,0016021 // integral component of membrane // i...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995_at,YLR326W,NaN,Saccharomyces cerevisiae,"Oct 6, 2014",Exemplar sequence,Saccharomyces Genome Database,YLR326W hypothetical protein,YLR326W,Putative protein of unknown function; predicte...,YLR326W,851036,NM_001182215,S000004318,NaN,0016020 // membrane // inferred from electroni...,NaN
9996_at,YLR327C,NaN,Saccharomyces cerevisiae,"Oct 6, 2014",Exemplar sequence,Saccharomyces Genome Database,YLR327C strong similarity to Stf2p,YLR327C,Protein of unknown function that associates wi...,TMA10,851037,NM_001182216,S000004319,NaN,0005634 // nucleus // inferred from direct ass...,NaN
9997_at,YLR328W,NaN,Saccharomyces cerevisiae,"Oct 6, 2014",Exemplar sequence,Saccharomyces Genome Database,YLR328W strong similarity to YGR010w,YLR328W,Nicotinic acid mononucleotide adenylyltransfer...,NMA1,851039,NM_001182217,S000004320,0009058 // biosynthetic process // inferred fr...,0005634 // nucleus // inferred from direct ass...,0000166 // nucleotide binding // inferred from...


In [4]:
cols = ["Sequence Source", "Target Description", "Gene Title", "Gene Ontology Biological Process", "Gene Ontology Cellular Component", "Gene Ontology Molecular Function"]

words = dict()
for row in metadata.index:
    words[row] = ""
    for col in cols:
        words[row] += " " + str(metadata.loc[row, col])


In [5]:
mdat_summary = pd.Series(words)
mdat_summary.name = "summary"
display(mdat_summary)

10000_at       Saccharomyces Genome Database YLR331C questio...
10001_at       Saccharomyces Genome Database YLR332W Protein...
10002_i_at     Saccharomyces Genome Database YLR333C Ribosom...
10003_f_at     Saccharomyces Genome Database YLR333C Ribosom...
10004_at       Saccharomyces Genome Database YLR334C questio...
                                    ...                        
9995_at        Saccharomyces Genome Database YLR326W hypothe...
9996_at        Saccharomyces Genome Database YLR327C strong ...
9997_at        Saccharomyces Genome Database YLR328W strong ...
9998_at        Saccharomyces Genome Database YLR329W 23 kDa ...
9999_at        Saccharomyces Genome Database YLR330W Involve...
Name: summary, Length: 9275, dtype: object

In [6]:
df = marker_labels.merge(mdat_summary, left_index=True, right_index=True)
display(df)

,labels,probabilities,summary
10001_at,7,0.857061,Saccharomyces Genome Database YLR332W Protein...
10003_f_at,3,0.999703,Saccharomyces Genome Database YLR333C Ribosom...
10005_at,3,0.997951,Saccharomyces Genome Database YLR335W nuclear...
10010_at,5,0.986816,Saccharomyces Genome Database YLR295C ATP syn...
10011_at,7,0.701117,Saccharomyces Genome Database YLR296W hypothe...
...,...,...,...
9992_at,0,0.993806,Saccharomyces Genome Database YLR323C weak si...
9994_at,3,0.998811,Saccharomyces Genome Database YLR325C Ribosom...
9996_at,2,0.966032,Saccharomyces Genome Database YLR327C strong ...
9998_at,2,0.993418,Saccharomyces Genome Database YLR329W 23 kDa ...


In [7]:
# get a list of all the words and frequency and remove stopwords and stuff
full_text = "\n".join(df["summary"].tolist())
full_text = full_text
print(len(full_text))

# remove everything but characters and numbers and whitespace -- this regex pattern was from AI
# pat = re.compile(r"[^a-zA-Z0-9\s]")
pat = re.compile(r"[^a-zA-Z0-9\s]")
full_text = re.sub(pat, "", full_text)
print(len(full_text))

# replace every whitespace with an actual space
pat = re.compile(r"[\s]+")
full_text = re.sub(pat, " ", full_text)
print(len(full_text))

# remove stopwords - figure out how to remove the ones with spaces
stopwords = ["inferred", "electronic", "annotation", "from", "of", "as", "for", "cell", "and", "the", "with", "a", "to", "has", "acts", "that", "arose", "assay", "similar", "on", "also", "at","author", "statement", "in", "involved", "annotation", "direct", "activity", "protein", "mutant", "phenotype", "interaction", "Saccharomyces", "process", "Genome", "database", "nan", "similarity", "physical", "or", "by", "sequence"]
stopwords += "between, identity, 100, GenBank, Found, yeast, SAGE, et, al, 1997, citation, VE, nonannotated, See, Characterization, Velculescu, 8243251, transcriptome, Cell".split(", ")
pat = re.compile(r'\b(' + '|'.join(re.escape(w) for w in stopwords) + r')\b *') # regex string written by Claude
full_text = re.sub(pat, " ", full_text)
print(len(full_text))

print(full_text[:5000])

# # remove single letters
# pat = re.compile(r"\sa-z\s")
# full_text = re.sub(pat, " ", full_text)
# print(len(full_text))

4843943
4495304
4360155
2645368
   Database YLR332W Protein required  mating Oglycosylated plasma membrane     sensor   wall integrity signaling  activates  pathway interacts  Rom2p  guanine nucleotide exchange factor  Rho1p    integrity pathway  Zeo1p MID2   paralog MTL1     whole genome duplication 0000767  morphogenesis   conjugation   genetic  0000767  morphogenesis   conjugation     0001101 response  acid     0006950 response  stress     0006970 response  osmotic stress   genetic  0030242 peroxisome degradation     0030969 UFPspecific transcription factor mRNA processing   endoplasmic reticulum unfolded  response     0031505 fungaltype  wall organization   genetic  0031505 fungaltype  wall organization     0005886 plasma membrane     0005887 integral component  plasma membrane     0016020 membrane     0016021 integral component  membrane     0043332 mating projection tip     0004888 transmembrane signaling receptor    genetic  0004888 transmembrane signaling receptor      0004888 

In [8]:
counts = pd.DataFrame(np.unique_counts(full_text.split())).T
counts.columns = ["Word", "Count"]
counts = counts.loc[counts["Count"] > 10].copy()
counts = counts.sort_values("Count").iloc[::-1]
counts = counts.reset_index(drop=True)
print(counts["Word"][:20])

display(counts)


0           binding
1          membrane
2          Database
3           complex
4               DNA
5         transport
6         cytoplasm
7           nucleus
8        regulation
9           genetic
10    transcription
11    mitochondrial
12          0005737
13              RNA
14          0005634
15       structural
16        component
17    mitochondrion
18          subunit
19          0005739
Name: Word, dtype: object


,Word,Count
0,binding,5747
1,membrane,4767
2,Database,3553
3,complex,2682
4,DNA,2587
...,...,...
2319,mediumchain,11
2320,catalase,11
2321,0006825,11
2322,0035267,11


In [9]:
reference_words = counts["Word"].tolist()
reference_dict = dict(zip([str(word) for word in reference_words], counts["Word"].index))

X = sparse.dok_array((len(df), len(reference_words)))

def clean(summary):
    # remove everything but characters and numbers and whitespace -- this regex pattern was from AI
    # pat = re.compile(r"[^a-zA-Z0-9\s]")
    pat = re.compile(r"[^a-zA-Z0-9\s]")
    summary = re.sub(pat, "", summary)
    # print(len(full_text))

    # replace every whitespace with an actual space
    pat = re.compile(r"[\s]+")
    summary = re.sub(pat, " ", summary)
    # print(len(summary))

    # remove stopwords - figure out how to remove the ones with spaces
    pat = re.compile(r'\b(' + '|'.join(re.escape(w) for w in stopwords) + r')\b *') # regex string written by Claude
    summary = re.sub(pat, " ", summary)

    # ALSO remove words that aren't in reference words - but this isn't working so I have another guard below
    pat = re.compile(r'\b(?!' + '|'.join(re.escape(w) for w in reference_words) + r'\b)\w+\b *') # also claude
    summary = re.sub(pat, " ", summary)

    return summary

mykeys = reference_dict.keys()

def bagcount(row, summary):
    summary = clean(summary)
    words = summary.split()
    for word in words:
        if word in mykeys:
            # print(row)
            # print(reference_dict[word])
            X[row, reference_dict[word]] += 1

for i, summary in enumerate(df["summary"]):
    bagcount(i, summary)

In [10]:
# try LDA - remove stopwords though
lda = LatentDirichletAllocation(n_components=10)
lda.fit(X)

,n_components,10
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


In [11]:
comps = lda.components_.T
topcomps = comps.argsort(axis=0)[::-1]
tops = topcomps[:20]
display(tops)

array([[ 37,  36,   0,  10,   1,   3,  11,   5,   0,  27],
       [ 29,   0,   7,  22,   2,  75,  17,   1,  24,  56],
       [ 18,   8,   6,   0,  57,   0,  19,  54,  71, 101],
       [ 32,  28,  42,  13,  58,  50,   1,  65,  76, 107],
       [ 15,   4,  12,   4,  16,  63,  77,  16,   4, 117],
       [ 80,   9,  43,   8,  60, 137,   0,  21,  20,  83],
       [ 69,  88,   3,  41,  21,   7,   2,  23,  35, 127],
       [112, 102,  14,  52,  73,  13,   3,  79, 108,   7],
       [  3,  94,  66,  31,  23, 105, 114,  78,  15, 135],
       [ 31,   2, 120,  82,  90, 178, 104,  84,  85,   0],
       [  0,  26,   2,   7,  34,  48,  35,  34, 119,  14],
       [  2,   6,  39, 111,  62,  14,  25,  70,   6,  18],
       [115,  12,   4,  59,  33,   2,  16,   9,  12, 184],
       [162,   7,  26,  14, 134, 157, 168,  30,  25,   3],
       [  6, 192,  47, 159,   6, 235, 177,   0, 132, 200],
       [ 12,  87, 110, 118,  64, 209, 171,   2, 141,  13],
       [179,  51,  50,  30,  25,  38, 185,  33, 147, 230

In [15]:
topwords = list()
for j in range(10):
    print(f"Topic {1 + j} top words:")
    topwords_message = ", ".join([reference_words[tops[i, j]] for i in range(len(tops))])
    topwords.append(topwords_message)
    print(topwords_message)

Topic 1 top words:
translation, ribosomal, subunit, ribosome, structural, constituent, large, 0003735, complex, factor, binding, Database, initiation, 0005840, cytoplasm, 0005737, translational, small, 0006412, Ribosomal
Topic 2 top words:
kinase, binding, regulation, cellular, DNA, genetic, bud, phosphorylation, repair, Database, response, cytoplasm, 0005737, nucleus, serinethreonine, cycle, transferase, neck, negative, 0005634
Topic 3 top words:
binding, nucleus, cytoplasm, catabolic, 0005737, ATP, complex, 0005634, hydrolase, ATPase, Database, stress, DNA, response, replication, 0016787, nuclear, mRNA, proteasome, 0005515
Topic 4 top words:
transcription, polymerase, binding, RNA, DNA, regulation, II, promoter, factor, DNAtemplated, nucleus, sequencespecific, positive, 0005634, orf, chromatin, ion, 0006351, response, genetic
Topic 5 top words:
membrane, Database, reticulum, endoplasmic, component, function, integral, unknown, 0016021, wall, 0016020, hypothetical, not, 0005783, cytop

In [17]:
display(comps.shape)
weights_frame = pd.DataFrame(comps, index=[reference_words], columns=[f"Topic {i+1}" for i in range(10)])

(2324, 10)

In [18]:
display(weights_frame)

,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7,Topic 8,Topic 9,Topic 10
binding,324.762196,790.856065,957.408969,1164.657617,72.162610,482.187756,474.440880,358.160695,809.094296,314.268916
membrane,0.100005,14.126523,1.089960,0.100005,1529.006545,0.100011,1108.668616,2114.608325,0.100007,0.100003
Database,314.339527,306.849553,329.383955,157.103016,1047.305033,207.746193,424.196437,328.672139,269.125586,169.278561
complex,367.599192,105.462623,546.998243,86.939100,34.396803,716.646203,394.386673,181.460526,0.708821,248.401815
DNA,16.693907,611.695631,314.754631,1011.826953,3.932989,78.599138,1.840603,17.508738,507.525452,23.621959
...,...,...,...,...,...,...,...,...,...,...
mediumchain,0.100000,0.100001,0.100000,0.100002,0.100018,0.100000,9.435383,1.764588,0.100009,0.100000
catalase,0.100000,0.100004,0.100012,0.100000,0.100000,0.100000,0.100029,11.099955,0.100000,0.100000
0006825,0.100000,0.100000,0.100000,0.100000,0.100001,0.100000,0.100023,11.099976,0.100000,0.100000
0035267,0.100002,0.100020,0.100000,0.100016,0.100000,0.100005,0.100000,0.100003,0.100000,11.099954


In [19]:
X_topics = lda.transform(X)

In [20]:
display(X_topics)

array([[1.11131683e-03, 4.60797538e-01, 3.19136659e-02, ...,
        2.74439618e-01, 1.11127169e-03, 1.11226571e-03],
       [9.76921772e-01, 2.56421672e-03, 2.56425362e-03, ...,
        2.56429742e-03, 2.56419045e-03, 2.56425661e-03],
       [5.55763133e-04, 5.55643270e-04, 4.74006273e-01, ...,
        1.57476634e-01, 5.55630288e-04, 4.56004687e-02],
       ...,
       [3.57919001e-01, 3.84704922e-03, 3.18914148e-01, ...,
        3.84643542e-03, 3.84683433e-03, 3.84659286e-03],
       [2.00023931e-03, 1.86495311e-01, 3.64835948e-02, ...,
        2.00019172e-03, 2.00049374e-03, 2.00025095e-03],
       [1.28230469e-03, 2.77064239e-01, 1.28258446e-03, ...,
        4.23825208e-01, 1.28241139e-03, 1.28224766e-03]])

In [24]:
topicdf = pd.DataFrame(X_topics, index=df.index.tolist(), columns=[f"Topic {i+1}" for i in range(10)])

In [25]:
display(topicdf)

,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7,Topic 8,Topic 9,Topic 10
10001_at,0.001111,0.460798,0.031914,0.022387,0.204904,0.001112,0.001111,0.274440,0.001111,0.001112
10003_f_at,0.976922,0.002564,0.002564,0.002564,0.002564,0.002564,0.002564,0.002564,0.002564,0.002564
10005_at,0.000556,0.000556,0.474006,0.085148,0.000556,0.208245,0.027301,0.157477,0.000556,0.045600
10010_at,0.000763,0.000763,0.754662,0.000763,0.000763,0.000763,0.151174,0.088820,0.000763,0.000763
10011_at,0.014286,0.014286,0.014286,0.014286,0.871421,0.014286,0.014287,0.014289,0.014286,0.014286
...,...,...,...,...,...,...,...,...,...,...
9992_at,0.001493,0.001493,0.001493,0.191895,0.001493,0.796162,0.001493,0.001493,0.001493,0.001494
9994_at,0.978048,0.002439,0.002439,0.002439,0.002439,0.002439,0.002439,0.002439,0.002439,0.002439
9996_at,0.357919,0.003847,0.318914,0.003847,0.237957,0.003847,0.062129,0.003846,0.003847,0.003847
9998_at,0.002000,0.186495,0.036484,0.070701,0.002001,0.694318,0.002000,0.002000,0.002000,0.002000


In [26]:
topicdf.to_csv("semantics.csv")
weights_frame.to_csv("semantic_weights.csv")

In [30]:
toptopics = pd.DataFrame(topwords, index=[f"Topic {i+1}" for i in range(10)], columns=["Top 10 Words"])
toptopics.to_csv("semantics_toptopics.csv")